In [1]:
from pathlib import Path
from typing import Callable

import numpy as np
import prism

from imagematerials.stock import compute_dynamic_stock_driven, compute_historic
from imagematerials.survival import ScipySurvival, SurvivalMatrix
from imagematerials.vehicles.preprocessing import (
    export_to_netcdf,
    import_from_netcdf,
    preprocessing,
)

In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema.nc")

In [3]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)

AttributeError: 'EntryPoints' object has no attribute 'get'

In [4]:
vehicle_nr = prep_data["simple"]['total_nr_vehicles_simple']
life_time_vehicles = prep_data["lifetimes"]

In [5]:
survival_matrix = SurvivalMatrix(ScipySurvival(life_time_vehicles))

In [6]:
start_simulation = 1970
end_simulation = vehicle_nr.coords["time"].max()
Region = prism.NewDim("region", coords=[str(x) for x in np.arange(1, 29)])
Mode = prism.NewDim("mode", coords=[str(x) for x in vehicle_nr["mode"].to_numpy()])
Cohort = prism.NewDim("cohort", coords=[str(x) for x in vehicle_nr["time"].to_numpy()])

In [7]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix
    stock: prism.TimeVariable[Region, Mode] #TODO check how to have property that can be both input and output within prism
    stock_function: Callable    # defines the stock function to use e.g. stock or inflow driven

    stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, "count"]         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.stock, self.survival_matrix, self.start_simulation,
                         self.stock_by_cohort, self.inflow, self.outflow_by_cohort,
                         self.stock_function)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        self.stock_function(self.stock, self.stock_by_cohort,  self.inflow, self.outflow_by_cohort, self.survival_matrix, t)

In [8]:
timeline = prism.Timeline(start=vehicle_nr.coords["time"][0],
                          end=end_simulation, stepsize=1)
timeline_simulate = prism.Timeline(start=start_simulation,
                          end=end_simulation, stepsize=1)

In [9]:
model = Stocks(timeline, start_simulation=start_simulation, survival_matrix=survival_matrix, stock=vehicle_nr, stock_function = compute_dynamic_stock_driven)

In [10]:
model.simulate(timeline_simulate)

c:\Users\Arp00003\AppData\Local\anaconda3\envs\fred_prism\Lib\site-packages\xarray\namedarray\core.py:215: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  return NamedArray(dims, np.asarray(data), attrs)
c:\Users\Arp00003\AppData\Local\anaconda3\envs\fred_prism\Lib\site-packages\xarray\namedarray\core.py:215: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  return NamedArray(dims, np.asarray(data), attrs)
c:\Users\Arp00003\AppData\Local\anaconda3\envs\fred_prism\Lib\site-packages\xarray\namedarray\core.py:215: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  return NamedArray(dims, np.asarray(data), attrs)
c:\Users\Arp00003\AppData\Local\anaconda3\envs\fred_prism\Lib\site-packages\xarray\namedarray\core.py:215: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  return NamedArray(dims, np.asarray(data), attrs)
c:\Users\Arp00003\Ap

In [18]:
import pandas as pd
bikes_stock_by_chort_old = pd.read_csv("bikes_by_cohort_vema.csv", index_col=[0,1])


In [78]:
bikes_stock_by_chort_old

time_cohort          Bikes
regions time                            
1       1900         1900       0.000000
        1900         1901       0.000000
        1900         1902       0.000000
        1900         1903       0.000000
        1900         1904       0.000000
...                   ...            ...
26      2060         2056  568692.915248
        2060         2057  562754.928682
        2060         2058  560865.025067
        2060         2059  562507.225987
        2060         2060  566243.751039

[673946 rows x 2 columns]

In [120]:
stock_cohort_year = model.stock_by_cohort[2000]



In [121]:
stock_cohort_year_bikes_new = stock_cohort_year.loc['Bikes', '1', 2000]
stock_cohort_year_bikes_new

<xarray.DataArray ()> Size: 8B
array(1008872.08333333)
Coordinates:
    mode     <U25 100B 'Bikes'
    region   <U2 8B '1'
    time     int64 8B 2000
    cohort   int64 8B 2000

In [117]:
test = bikes_stock_by_chort_old.reset_index()
test = test.set_index(['regions','time','time_cohort'])

In [124]:
test.loc[1, 2000,2000]



Bikes    0.0
Name: (1, 2001, 2000), dtype: float64

In [119]:
2857292.98255053/113748.324135

25.11942926877241